In [29]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader
import os
import sys
from sklearn.model_selection import train_test_split
# # 获取当前脚本所在目录的绝对路径（即子目录 scripts 的路径）
# current_dir = os.path.dirname(os.path.abspath('/data/huangjh/code/Projects_PINN_LiB/PIKAN4SOH/Notebooks/MIT_Dataloader.ipynb'))
# # 获取上级目录路径（即 project_root/utils 的父目录 project_root）
# parent_dir = os.path.dirname(current_dir)
# # 将上级目录添加到模块搜索路径
# sys.path.append(parent_dir)
# from utils.util import write_to_txt
device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu" )
# device = 'cpu'
print(device)

cuda:3


In [30]:
class DF_MIT():
    def __init__(self,args):
        self.normalization = True
        self.normalization_method = args.normalization_method # min-max, z-score
        self.args = args

    def _3_sigma(self, Ser1):
        rule = (Ser1.mean() - 3 * Ser1.std() > Ser1) | (Ser1.mean() + 3 * Ser1.std() < Ser1)
        index = np.arange(Ser1.shape[0])[rule]
        return index

    def delete_3_sigma(self,df):
        df = df.replace([np.inf, -np.inf], np.nan)
        df = df.dropna()
        df = df.reset_index(drop=True)
        out_index = []
        for col in df.columns:
            index = self._3_sigma(df[col])
            out_index.extend(index)
        out_index = list(set(out_index))
        df = df.drop(out_index, axis=0)
        df = df.reset_index(drop=True)
        return df

    def read_one_csv(self,file_name,nominal_capacity=None):
        df = pd.read_csv(file_name)
        df.insert(df.shape[1]-1,'cycle index',np.arange(df.shape[0]))

        df = self.delete_3_sigma(df)

        if nominal_capacity is not None:
            #print(f'nominal_capacity:{nominal_capacity}, capacity max:{df["capacity"].max()}',end=',')
            df['capacity'] = df['capacity']/nominal_capacity
            #print(f'SOH max:{df["capacity"].max()}')
            f_df = df.iloc[:,:-1]
            if self.normalization_method == 'min-max':
                f_df = 2*(f_df - f_df.min())/(f_df.max() - f_df.min()) - 1
            elif self.normalization_method == 'z-score':
                f_df = (f_df - f_df.mean())/f_df.std()

            df.iloc[:,:-1] = f_df

        return df

    def load_one_battery(self,path,nominal_capacity=None):
        df = self.read_one_csv(path,nominal_capacity)
        # MIT数据特征选择
        # Var2：'CC Q','cycle index'
        # Var3：'CC Q','voltage entropy','cycle index'
        # Var4：'voltage mean','CC Q','voltage entropy','cycle index'
        df = df.filter(items=['CC Q','cycle index','capacity']) # CC Q在恒流充电时等价于CC charge time
        x = df.iloc[:,:-1].values
        y = df.iloc[:,-1].values
        x1 = x[:-1]
        x2 = x[1:]
        y1 = y[:-1]
        y2 = y[1:]
        return (x,y),(x1,y1),(x2,y2)

    def load_all_battery(self,path_list,nominal_capacity):
        X, Y, X1, X2, Y1, Y2 = [], [], [], [], [], []
        # if self.args.log_dir is not None and self.args.save_folder is not None:
        #     save_name = os.path.join(self.args.save_folder,self.args.log_dir)
        #     write_to_txt(save_name,'data path:')
        #     write_to_txt(save_name,str(path_list))
        for path in path_list:
            (x, y),(x1, y1), (x2, y2) = self.load_one_battery(path, nominal_capacity)
            X.append(x)
            X1.append(x1)
            X2.append(x2)
            Y.append(y)
            Y1.append(y1)
            Y2.append(y2)

        X = np.concatenate(X, axis=0)
        X1 = np.concatenate(X1, axis=0)
        X2 = np.concatenate(X2, axis=0)
        Y = np.concatenate(Y, axis=0)
        Y1 = np.concatenate(Y1, axis=0)
        Y2 = np.concatenate(Y2, axis=0)

        tensor_X = torch.from_numpy(X).float().to(device) 
        tensor_X1 = torch.from_numpy(X1).float().to(device)
        tensor_X2 = torch.from_numpy(X2).float().to(device)
        tensor_Y = torch.from_numpy(Y).float().view(-1,1).to(device)
        tensor_Y1 = torch.from_numpy(Y1).float().view(-1,1).to(device)
        tensor_Y2 = torch.from_numpy(Y2).float().view(-1,1).to(device)

        train_X1, valid_X1, train_X2, valid_X2, train_Y1, valid_Y1, train_Y2, valid_Y2 = \
            train_test_split(tensor_X1, tensor_X2, tensor_Y1, tensor_Y2, test_size=0.2, random_state=420)
        train_loader = DataLoader(TensorDataset(train_X1, train_X2, train_Y1, train_Y2),
                                  batch_size=self.args.batch_size,
                                  shuffle=True)
        valid_loader = DataLoader(TensorDataset(valid_X1, valid_X2, valid_Y1, valid_Y2),
                                  batch_size=self.args.batch_size,
                                  shuffle=True)
        test_loader = DataLoader(TensorDataset(tensor_X1, tensor_X2, tensor_Y1, tensor_Y2),
                                 batch_size=self.args.batch_size,
                                 shuffle=False)

        data = {'input': tensor_X, 'label': tensor_Y,
                  'train_loader': train_loader,
                  'valid_loader': valid_loader,
                  'test_loader': test_loader}

        return data

In [31]:
class MITdataFilter(DF_MIT):
    def __init__(self,root='../data/MIT data',args=None):
        super(MITdataFilter, self).__init__(args)
        self.root = root
        self.batchs = ['2017-05-12','2017-06-30','2018-04-12']
        if self.normalization:
            self.nominal_capacity = 1.1
        else:
            self.nominal_capacity = None

    def read_one_batch(self,batch):
        '''
        读取一个批次的csv文件
        English version: Read a batch of csv files
        :param batch: int,可选[1,2,3]
        :return: dict
        '''
        assert batch in [1,2,3], 'batch must be in {}'.format([1,2,3])
        root = os.path.join(self.root,self.batchs[batch-1])
        file_list = os.listdir(root)
        path_list = []
        for file in file_list:
            file_name = os.path.join(root,file)
            path_list.append(file_name)
        return self.load_all_battery(path_list=path_list, nominal_capacity=self.nominal_capacity)

    def read_all(self,specific_path_list=None):
        '''
        读取所有csv文件。如果指定了specific_path_list,则读取指定的文件；否则读取所有文件；封装成dataloader
        English version:
        Read all csv files.
        If specific_path_list is not None, read the specified file; otherwise read all files;
        :param self:
        :return: dict
        '''
        if specific_path_list is None:
            file_list = []
            for batch in self.batchs:
                root = os.path.join(self.root,batch)
                files = os.listdir(root)
                for file in files:
                    path = os.path.join(root,file)
                    file_list.append(path)
            return self.load_all_battery(path_list=file_list, nominal_capacity=self.nominal_capacity)
        else:
            return self.load_all_battery(path_list=specific_path_list, nominal_capacity=self.nominal_capacity)

In [32]:
def load_MIT_data_filter(args,small_sample=None):   # 无差分，差分：diff
    root = '../data/MIT data'
    train_list = []
    test_list = []
    for batch in ['2017-05-12','2017-06-30','2018-04-12']:
        batch_root = os.path.join(root,batch)
        files = os.listdir(batch_root)
        for f in files:
            id = int(f.split('-')[-1].split('.')[0])
            if id % 5 == 0:
                test_list.append(os.path.join(batch_root,f))
            else:
                train_list.append(os.path.join(batch_root,f))
    if small_sample is not None:
        train_list = train_list[:small_sample]
    data = MITdataFilter(root=root,args=args)
    train_data = data.read_all(specific_path_list=train_list)
    test_data = data.read_all(specific_path_list=test_list)
    
    dataset = {'train_input':train_data['input'],
                  'train_label':train_data['label'],
                  'test_input':test_data['input'],
                  'test_label':test_data['label'],
                  'train_loader':train_data['train_loader'],
                  'valid_loader':train_data['valid_loader'],
                  'test_loader':test_data['test_loader']}
    return dataset

In [33]:
import argparse
parser = argparse.ArgumentParser('存储过滤出关键健康指标的MIT数据集')
parser.add_argument('--dataset',type=str,default='MIT',choices=['XJTU','HUST','MIT','TJU'])
parser.add_argument('--data_root', type=str, default='../data/MIT data', help='MIT数据集根路径')
parser.add_argument('--normalization_method',type=str, default='min-max', help='min-max,z-score')
parser.add_argument('--batch_size',type=int,default=512)
args, _ = parser.parse_known_args()

In [34]:
dataset = load_MIT_data_filter(args)
torch.save(dataset, '../Data/MIT_Data_Var2.pt')

/tmp/ipykernel_1294411/774110614.py:41: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0     -1.000000
1     -0.997033
2     -0.994065
3     -0.991098
4     -0.988131
         ...   
666    0.976261
667    0.979228
668    0.982196
669    0.997033
670    1.000000
Name: cycle index, Length: 671, dtype: float64' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.iloc[:,:-1] = f_df
/tmp/ipykernel_1294411/774110614.py:41: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0     -1.000000
1     -0.995614
2     -0.993421
3     -0.991228
4     -0.989035
         ...   
903    0.986842
904    0.989035
905    0.991228
906    0.995614
907    1.000000
Name: cycle index, Length: 908, dtype: float64' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.iloc[:,:-1] = f_df
/tmp/ipykernel

In [35]:
import gc
torch.cuda.empty_cache()
gc.collect()

73